In [ ]:
import numpy as np
import torch
import onnx
import hls4ml
from torch.utils.data import DataLoader
from utils.utils import read_object
from hls4ml.model.profiling import numerical
from onnx import numpy_helper
from cnn_base import t1_quantizedCNN
from sklearn.metrics import accuracy_score
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.onnx_exec import execute_onnx

In [ ]:
onnx_model_path = "./hls4ml_onnxs/sat6-cnn-t1w8.onnx"
pytorch_model_path = 'pytorch_models/sat6-cnn-t1.pt'
batch_size = 512
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_dataset_output_name = "./dataset/test_dataset.pkl"

dataset_test = read_object(test_dataset_output_name)
loader_test = DataLoader(dataset_test, batch_size, shuffle=False)

X_test = dataset_test.tensors[0].numpy().astype(np.float32)
y_test = dataset_test.tensors[1].numpy().astype(np.int32)

x = X_test[0:1]

if isinstance(x, np.ndarray):
    x_torch = torch.from_numpy(x).float()
else:
    x_torch = x.float()

print("Input shape:", x_torch.shape)


In [ ]:
model = t1_quantizedCNN(8)
model.load_state_dict(torch.load(pytorch_model_path, map_location=device))
model.eval()

with torch.no_grad():
    y_brevitas = model(x_torch)

y_brevitas = y_brevitas.detach().cpu().numpy()

print("\n=== BREVITAS ===")
print("shape:", y_brevitas.shape)
print("output:", y_brevitas)
print("prediction:", np.argmax(y_brevitas, axis=1))
print("ground truth:", y_test[0:1])

In [ ]:
qonnx_model = ModelWrapper(onnx_model_path)

input_name = qonnx_model.graph.input[0].name
output_name = qonnx_model.graph.output[0].name

print("\n=== QONNX ===")
print("input name :", input_name)
print("output name:", output_name)


input_dict = {
    input_name: x.transpose(0, 2, 3, 1)
}
output_dict = execute_onnx(
    qonnx_model,
    input_dict
)

y_qonnx = output_dict[output_name]

print("shape:", y_qonnx.shape)
print("output:", y_qonnx)
print("prediction:", np.argmax(y_qonnx, axis=1))

In [ ]:
diff = y_brevitas - y_qonnx

print("\n=== COMPARAÇÃO ===")
print("max abs diff :",
      np.max(np.abs(diff)))
print("mean abs diff:",
      np.mean(np.abs(diff)))
print("Brevitas prediction:",
      np.argmax(y_brevitas, axis=1))
print("QONNX prediction:",
      np.argmax(y_qonnx, axis=1))
print(
    "Predição igual:",
    np.array_equal(
        np.argmax(y_brevitas, axis=1),
        np.argmax(y_qonnx, axis=1)
    )
)

In [ ]:
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.onnx_exec import execute_onnx
import numpy as np


qonnx_model = ModelWrapper(onnx_model_path)

# ============================================================
# Inputs / outputs
# ============================================================

input_name = qonnx_model.graph.input[0].name

input_dict = {
    input_name: x.numpy()
}


# ============================================================
# Executar QONNX
# ============================================================

output_dict = execute_onnx(
    qonnx_model,
    input_dict,
    return_full_exec_context=True
)


# ============================================================
# Mostrar tensores relevantes
# ============================================================

tensors_to_check = [
    '98LuDZ',       # Conv_0 output
    'Quant_3_out0', # Relu_0 output
    '0QMCxl',       # Quant_3 output
    'ZFueGr',       # Conv_1 output
    'Conv_1_out0',  # Relu_1 output
    'Relu_1_out0',  # Quant_4 output
    '1ib8ok',       # Flatten
    'global_out'    # MatMul
]

for tensor_name in tensors_to_check:

    if tensor_name not in output_dict:
        print(f"\n{tensor_name}: NÃO ENCONTRADO")
        continue

    value = output_dict[tensor_name]

    print("\n" + "=" * 60)
    print(tensor_name)
    print("=" * 60)

    print("shape:", value.shape)
    print("dtype:", value.dtype)
    print("min  :", np.min(value))
    print("max  :", np.max(value))
    print("mean :", np.mean(value))
    print("std  :", np.std(value))